# Task 2: Data Preprocessing & Augmentation
## Real-Time Face Segmentation for Movie Cast Identification

In this notebook we:
1. Load raw data and generated masks from Task 1
2. Standardize all images to 3 channels (RGB)
3. Resize images and masks to 256×256
4. Normalize pixel values to [0, 1]
5. Train/Validation split (80/20)
6. Data augmentation (flip, rotate, brightness)
7. Save preprocessed data for model training

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
import os

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE = 256
SEED = 42
np.random.seed(SEED)

## 2.1 Load Raw Data

In [ ]:
# Load original data and masks from Task 1
data = np.load('Part 1- Train data - images.npy', allow_pickle=True)
all_masks = np.load('all_masks.npy', allow_pickle=True)

print(f'Total samples: {len(data)}')
print(f'Total masks: {len(all_masks)}')

## 2.2 Standardize Channels, Resize & Normalize

In [ ]:
def preprocess_image(img, target_size=IMG_SIZE):
    """Convert to RGB, resize to target_size x target_size, normalize to [0,1]."""
    # Standardize to 3 channels
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    
    # Resize
    img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    
    # Normalize to [0, 1]
    img = img.astype(np.float32) / 255.0
    return img


def preprocess_mask(mask, target_size=IMG_SIZE):
    """Resize mask to target_size x target_size using nearest neighbor (preserves binary values)."""
    mask = cv2.resize(mask, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
    mask = mask.astype(np.float32)
    return mask


# Process all images and masks
print(f'Preprocessing {len(data)} images to {IMG_SIZE}x{IMG_SIZE}...')

images = []
masks = []

for i in range(len(data)):
    img = preprocess_image(data[i][0])
    msk = preprocess_mask(all_masks[i])
    images.append(img)
    masks.append(msk)

images = np.array(images)
masks = np.array(masks)

# Add channel dimension to masks: (N, 256, 256) -> (N, 256, 256, 1)
masks = np.expand_dims(masks, axis=-1)

print(f'Images shape: {images.shape}, dtype: {images.dtype}')
print(f'Masks shape:  {masks.shape}, dtype: {masks.dtype}')
print(f'Image pixel range: [{images.min():.2f}, {images.max():.2f}]')
print(f'Mask unique values: {np.unique(masks)}')

## 2.3 Verify Preprocessing — Visual Check

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
sample_idx = np.random.choice(len(images), 6, replace=False)

for i, idx in enumerate(sample_idx):
    row = i // 2
    col = (i % 2) * 2
    
    axes[row, col].imshow(images[idx])
    axes[row, col].set_title(f'Image {idx} (256x256)')
    axes[row, col].axis('off')
    
    axes[row, col + 1].imshow(masks[idx, :, :, 0], cmap='gray')
    axes[row, col + 1].set_title(f'Mask {idx}')
    axes[row, col + 1].axis('off')

plt.suptitle('Preprocessed Images & Masks (256x256, Normalized)', fontsize=14)
plt.tight_layout()
plt.show()

## 2.4 Train / Validation Split (80/20)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    images, masks, test_size=0.2, random_state=SEED
)

print(f'Training set:   {X_train.shape[0]} images')
print(f'Validation set: {X_val.shape[0]} images')
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_val shape:   {X_val.shape}')
print(f'y_val shape:   {y_val.shape}')

## 2.5 Data Augmentation

We apply augmentations to the training set to improve model generalization:
- Horizontal flip
- Random rotation (±15°)
- Random brightness adjustment

Augmentations are applied identically to both image and mask to keep them aligned.

In [ ]:
def augment_pair(image, mask):
    """Apply random augmentations to an image-mask pair."""
    # Random horizontal flip
    if np.random.random() > 0.5:
        image = np.fliplr(image)
        mask = np.fliplr(mask)
    
    # Random rotation (±15 degrees)
    if np.random.random() > 0.5:
        angle = np.random.uniform(-15, 15)
        h, w = image.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        image = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
        # Re-add channel dim if lost
        if len(mask.shape) == 2:
            mask = np.expand_dims(mask, axis=-1)
    
    # Random brightness adjustment (image only, not mask)
    if np.random.random() > 0.5:
        factor = np.random.uniform(0.7, 1.3)
        image = np.clip(image * factor, 0.0, 1.0)
    
    return image, mask


# Generate augmented training data
print('Generating augmented training data...')
aug_images = []
aug_masks = []

# Keep all originals
for i in range(len(X_train)):
    aug_images.append(X_train[i])
    aug_masks.append(y_train[i])

# Add augmented copies (1 augmented version per original = 2x training data)
for i in range(len(X_train)):
    aug_img, aug_msk = augment_pair(X_train[i].copy(), y_train[i].copy())
    aug_images.append(aug_img.astype(np.float32))
    aug_masks.append(aug_msk.astype(np.float32))

X_train_aug = np.array(aug_images)
y_train_aug = np.array(aug_masks)

print(f'Original training set: {X_train.shape[0]} images')
print(f'Augmented training set: {X_train_aug.shape[0]} images')
print(f'X_train_aug shape: {X_train_aug.shape}')
print(f'y_train_aug shape: {y_train_aug.shape}')

## 2.6 Visualize Augmented Samples

In [ ]:
# Show original vs augmented pairs
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(3):
    idx = np.random.randint(0, len(X_train))
    orig_img = X_train[idx]
    orig_mask = y_train[idx]
    aug_img, aug_mask = augment_pair(orig_img.copy(), orig_mask.copy())
    
    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_title('Original Image')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(orig_mask[:, :, 0], cmap='gray')
    axes[i, 1].set_title('Original Mask')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(np.clip(aug_img, 0, 1))
    axes[i, 2].set_title('Augmented Image')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(aug_mask[:, :, 0], cmap='gray')
    axes[i, 3].set_title('Augmented Mask')
    axes[i, 3].axis('off')

plt.suptitle('Original vs Augmented (Flip + Rotate + Brightness)', fontsize=14)
plt.tight_layout()
plt.show()

## 2.7 Create tf.data Pipelines for Efficient Training

In [ ]:
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Training dataset with shuffling
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_aug, y_train_aug))
train_dataset = train_dataset.shuffle(buffer_size=1000, seed=SEED)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(AUTOTUNE)

# Validation dataset (no shuffle, no augmentation)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.prefetch(AUTOTUNE)

# Verify
for batch_img, batch_mask in train_dataset.take(1):
    print(f'Train batch - images: {batch_img.shape}, masks: {batch_mask.shape}')

for batch_img, batch_mask in val_dataset.take(1):
    print(f'Val batch   - images: {batch_img.shape}, masks: {batch_mask.shape}')

print(f'\nTotal training batches: {len(train_dataset)}')
print(f'Total validation batches: {len(val_dataset)}')

## 2.8 Save Preprocessed Data

In [ ]:
# Save for Task 3 (Model Building)
np.save('X_train_aug.npy', X_train_aug)
np.save('y_train_aug.npy', y_train_aug)
np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)

print('Saved preprocessed data:')
print(f'  X_train_aug.npy: {X_train_aug.shape} ({X_train_aug.nbytes / 1e6:.1f} MB)')
print(f'  y_train_aug.npy: {y_train_aug.shape} ({y_train_aug.nbytes / 1e6:.1f} MB)')
print(f'  X_val.npy:       {X_val.shape} ({X_val.nbytes / 1e6:.1f} MB)')
print(f'  y_val.npy:       {y_val.shape} ({y_val.nbytes / 1e6:.1f} MB)')
print('\nTask 2 Complete! Ready for Task 3: Model Building (U-Net + MobileNetV2)')

## Summary

| Step | Details |
|------|--------|
| Channel standardization | Grayscale→RGB, RGBA→RGB |
| Resize | All images & masks → 256×256 |
| Normalization | Pixels scaled to [0, 1] |
| Train/Val split | 80/20 (327 train, 82 val) |
| Augmentation | Horizontal flip, ±15° rotation, brightness (0.7-1.3x) |
| Augmented training size | ~654 images (2x original) |
| Batch size | 16 |
| Data format | tf.data pipelines with prefetching |